# 05: Choropleth Maps

This notebook demonstrates choropleth map creation using `siege_utilities.geo.choropleth`:

1. **Simple Choropleths** — `create_choropleth()` for single-variable maps
2. **Multi-Variable Comparison** — `create_choropleth_comparison()` for side-by-side maps
3. **Classification Schemes** — `create_classified_comparison()` with quantiles, equal interval, etc.
4. **Bivariate Choropleths** — `create_bivariate_choropleth()` for two variables simultaneously
5. **Using ChartGenerator** — Choropleth methods from the reporting system
6. **Saving Maps** — `save_map()` for export to PNG/PDF/SVG
7. **3D Map Rendering** — `ThreeDMapRenderer` from `reporting.map_3d`, pydeck hexbin/column layers

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[geo]
# mapclassify is included in the geo extras for classification schemes
# For 3D maps: pip install -e /path/to/siege_utilities[3d]
```

In [ ]:
%matplotlib inline

# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import siege_utilities as su
from siege_utilities.geo import get_census_boundaries
from siege_utilities.geo.choropleth import (
    create_choropleth,
    create_choropleth_comparison,
    create_classified_comparison,
    create_bivariate_choropleth,
    create_bivariate_crosstab,
    create_bivariate_companion_maps,
    verify_bivariate_classification,
    create_bivariate_analysis,
    BivariateAnalysisResult,
    save_map,
    BIVARIATE_COLOR_SCHEMES,
    SCHEME_LABELS,
)

import numpy as np
import matplotlib.pyplot as plt

# Initialize logging
su.configure_shared_logging(level="INFO")

# Set random seed for reproducibility
np.random.seed(42)

su.log_info("Imports successful!")
su.log_info(f"Available bivariate color schemes: {list(BIVARIATE_COLOR_SCHEMES.keys())}")
su.log_info(f"Available classification schemes: {list(SCHEME_LABELS.keys())}")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_05"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Branding configuration ---
from siege_utilities.reporting.client_branding import ClientBrandingManager
_branding_mgr = ClientBrandingManager()
BRANDING = _branding_mgr.get_client_branding('siege_analytics')
COLORS = BRANDING['colors']

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
su.log_info(f"Branding: {BRANDING['name']} (primary={COLORS['primary']})")

## 1. Load Geographic Data

We'll use California counties as our base geography.

In [ ]:
# Download California counties
counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')
ca_counties = counties[counties['statefp'] == '06'].copy()

su.log_info(f"Loaded {len(ca_counties)} California counties")
su.log_info(f"Columns: {list(ca_counties.columns)[:8]}...")

In [ ]:
# Create sample data - simulated population and income
ca_counties['population'] = np.random.randint(10000, 10000000, size=len(ca_counties))
ca_counties['median_income'] = np.random.randint(40000, 150000, size=len(ca_counties))
ca_counties['unemployment_rate'] = np.random.uniform(2.0, 15.0, size=len(ca_counties))

# Calculate area in square miles
ca_counties['area_sq_mi'] = ca_counties['aland'] / 2_589_988
ca_counties['pop_density'] = ca_counties['population'] / ca_counties['area_sq_mi']

su.log_info("Sample data:")
su.log_info(str(ca_counties[['name', 'population', 'median_income', 'unemployment_rate']].head()))

## 2. Simple Choropleth with GeoPandas

The most straightforward way to create a choropleth using GeoDataFrame.plot()

In [ ]:
# Simple choropleth - population
fig, ax = create_choropleth(
    ca_counties,
    column='population',
    title='California Population by County',
    cmap='YlOrRd',
    legend_kwds={'label': 'Population', 'shrink': 0.6},
)
save_map(fig, str(OUTPUT_DIR / 'choropleth_population.png'))
plt.show()
plt.close()

In [ ]:
# Compare multiple variables side by side
fig, axes = create_choropleth_comparison(
    ca_counties,
    columns=[
        {'column': 'median_income', 'title': 'Median Income (Sequential)', 'cmap': 'Blues'},
        {'column': 'unemployment_rate', 'title': 'Unemployment Rate (Diverging)', 'cmap': 'RdYlGn_r'},
        {'column': 'pop_density', 'title': 'Population Density', 'cmap': 'Purples'},
    ],
    figsize=(18, 8),
    ncols=3,
)
save_map(fig, str(OUTPUT_DIR / 'comparison_3panel.png'))
plt.show()
plt.close()

## 3. Classification Schemes

Different ways to bin continuous data for choropleth maps.

In [ ]:
# Compare classification schemes for population
fig, axes = create_classified_comparison(
    ca_counties,
    column='population',
    schemes=['quantiles', 'equal_interval', 'fisher_jenks', 'percentiles'],
    k=5,
    cmap='YlOrRd',
    figsize=(14, 14),
    ncols=2,
)
save_map(fig, str(OUTPUT_DIR / 'classified_comparison.png'))
plt.show()
plt.close()

## 4. Bivariate Choropleth

Show two variables simultaneously using a 2D color scheme.

In [ ]:
# Bivariate choropleth: Population vs Median Income
fig, axes = create_bivariate_choropleth(
    ca_counties,
    var1='population',
    var2='median_income',
    title='California: Population vs Median Income by County',
)
save_map(fig, str(OUTPUT_DIR / 'bivariate_pop_income.png'))
plt.show()
plt.close()

In [ ]:
# Bivariate choropleth with a different color scheme
fig, axes = create_bivariate_choropleth(
    ca_counties,
    var1='pop_density',
    var2='unemployment_rate',
    title='California: Population Density vs Unemployment Rate',
    color_scheme='teuling',
)
save_map(fig, str(OUTPUT_DIR / 'bivariate_density_unemployment.png'))
plt.show()
plt.close()

## 4b. Bivariate Verification

The bivariate choropleth is powerful but hard to audit visually. These verification tools let you:

1. **Crosstab** — see how many geographic units fall in each class pair
2. **Companion maps** — see each variable independently with matching quantile bins
3. **Verification check** — confirm classification integrity (no missing units, monotonic breakpoints)
4. **Full analysis** — one call to produce all artifacts at once

In [ ]:
# 4b-1: Crosstab — how many counties in each bivariate class pair?
# The rendered version colors cells to match the bivariate legend.
ct, ct_fig, ct_ax = create_bivariate_crosstab(
    ca_counties, 'population', 'median_income',
    render=True,
)
save_map(ct_fig, str(OUTPUT_DIR / 'bivariate_crosstab.png'))
plt.show()
plt.close()

# Also display the raw DataFrame
su.log_info("Bivariate crosstab (counts per class pair):")
display(ct)

In [ ]:
# 4b-2: Companion maps — each variable shown independently with same quantile bins
comp_fig, comp_axes = create_bivariate_companion_maps(
    ca_counties, 'population', 'median_income',
)
save_map(comp_fig, str(OUTPUT_DIR / 'bivariate_companion_maps.png'))
plt.show()
plt.close()

In [ ]:
# 4b-3: Verification — confirm classification integrity
v = verify_bivariate_classification(
    ca_counties, 'population', 'median_income',
)

su.log_info(f"Classification valid: {v['valid']}")
su.log_info(f"Total units: {v['total_units']}")
su.log_info(f"Classified: {v['classified_units']}")
su.log_info(f"Var1 breakpoints: {v['var1_breaks']}")
su.log_info(f"Var2 breakpoints: {v['var2_breaks']}")
if v['errors']:
    for err in v['errors']:
        su.log_warning(f"  ERROR: {err}")
else:
    su.log_info("All checks passed!")

In [ ]:
# 4b-4: Full analysis — one call produces everything
result = create_bivariate_analysis(
    ca_counties, 'pop_density', 'unemployment_rate',
    title='Pop Density vs Unemployment: Full Analysis',
    color_scheme='teuling',
)

# Show the bivariate map (legend now includes magnitude tick labels)
save_map(result.bivariate_fig, str(OUTPUT_DIR / 'analysis_bivariate.png'))
plt.show()
plt.close(result.bivariate_fig)

# Show the crosstab
su.log_info("Crosstab:")
display(result.crosstab)

# Show companion maps
save_map(result.companion_fig, str(OUTPUT_DIR / 'analysis_companion.png'))
plt.show()
plt.close(result.companion_fig)

# Show verification
su.log_info(f"Verification passed: {result.verification['valid']}")
su.log_info(f"Var1 breaks: {result.var1_breaks}")
su.log_info(f"Var2 breaks: {result.var2_breaks}")

## 5. Using ChartGenerator

The ChartGenerator class provides choropleth methods designed for integration with the reporting system.

In [ ]:
# Initialize ChartGenerator (from reporting module) with branding
try:
    from siege_utilities.reporting.chart_generator import ChartGenerator
    chart_gen = ChartGenerator(branding_config=BRANDING, output_dir=OUTPUT_DIR)

    # Prepare data for ChartGenerator (expects location column and value column)
    chart_data = ca_counties[['name', 'population', 'median_income']].copy()
    chart_data = chart_data.rename(columns={'name': 'county'})

    su.log_info(f"ChartGenerator initialized with {BRANDING['name']} branding")
    su.log_info("Available methods: create_choropleth_map, create_bivariate_choropleth")
except ImportError as e:
    su.log_warning(f"ChartGenerator not available: {e}")
    su.log_warning("Skipping ChartGenerator demos — install reporting extras if needed")

In [ ]:
# Check if Folium is available (required for ChartGenerator maps)
try:
    import folium
    HAS_FOLIUM = True
    su.log_info(f"Folium version: {folium.__version__}")
except ImportError:
    HAS_FOLIUM = False
    su.log_warning("Folium not available - ChartGenerator maps will use fallback")

## 6. Saving Maps

Export choropleth maps to various formats.

In [ ]:
# Create and save a choropleth to PNG
fig, ax = create_choropleth(
    ca_counties,
    column='median_income',
    title='California Median Income by County',
    cmap='Greens',
    legend_kwds={
        'label': 'Median Income ($)',
        'shrink': 0.6,
        'format': '${x:,.0f}',
    },
    figsize=(12, 14),
)

output_path = save_map(fig, str(OUTPUT_DIR / 'ca_income_choropleth.png'))
su.log_info(f"Saved to: {output_path}")
plt.show()
plt.close()

## 7. 3D Map Rendering with pydeck

The `reporting.map_3d` module provides `ThreeDMapRenderer` for creating
interactive 3D maps using pydeck (deck.gl). It supports:

- **Hexagonal binning** — aggregate point data into 3D hex columns
- **Column layers** — one column per data point, height proportional to value
- **3D choropleths** — extruded polygons from a GeoDataFrame
- **HTML export** and **in-notebook rendering**

This complements the 2D choropleth workflow above with a third dimension
for density or magnitude, useful for donation hotspots, voter density, or
event clustering.

```bash
pip install -e /path/to/siege_utilities[3d]
# pydeck is included in the 3d extras
```

In [ ]:
try:
    from siege_utilities.reporting.map_3d import (
        ThreeDMapRenderer,
        create_3d_hexbin,
        create_3d_columns,
        MAP_STYLES,
        PYDECK_AVAILABLE,
    )
    HAS_MAP3D = True
except ImportError as e:
    HAS_MAP3D = False
    su.log_warning(f"map_3d import failed ({e}) — install 3d extras if needed.")

if HAS_MAP3D:
    su.log_info("Available map styles:")
    for name, url in MAP_STYLES.items():
        su.log_info(f"  {name:<12} {url}")

    su.log_info(f"pydeck installed: {PYDECK_AVAILABLE}")

In [ ]:
# Generate sample point data — simulated donation locations across California
if HAS_MAP3D and PYDECK_AVAILABLE:
    import pandas as pd

    rng = np.random.default_rng(42)
    n = 500
    sample_points = pd.DataFrame({
        "latitude": 36.7 + rng.normal(0, 1.5, n),
        "longitude": -119.5 + rng.normal(0, 1.2, n),
        "value": rng.integers(100, 5000, n),
    })

    su.log_info(f"Sample point data: {len(sample_points)} simulated donations")
    su.log_info(str(sample_points.head()))

    # --- Hexagonal binning layer ---
    renderer = ThreeDMapRenderer(map_style="dark")
    deck_hex = renderer.create_hexagon_layer(
        sample_points, radius=500, elevation_scale=50, zoom=11,
    )
    su.log_info(f"Hexbin deck created: {type(deck_hex).__name__}")

    # --- Column layer ---
    deck_cols = renderer.create_column_layer(
        sample_points, value_col="value", radius=200,
        elevation_scale=1, color=[30, 144, 255, 180], zoom=12,
    )
    su.log_info(f"Column deck created: {type(deck_cols).__name__}")

    # --- Export to HTML ---
    hex_html = renderer.render_to_html(deck_hex, str(OUTPUT_DIR / '3d_hexbin_map.html'))
    col_html = renderer.render_to_html(deck_cols, str(OUTPUT_DIR / '3d_column_map.html'))

    su.log_info(f"Hexbin HTML: {hex_html.name} ({hex_html.stat().st_size:,} bytes)")
    su.log_info(f"Column HTML: {col_html.name} ({col_html.stat().st_size:,} bytes)")

elif HAS_MAP3D:
    su.log_warning("pydeck not installed — install with: pip install 'siege-utilities[3d]'")
else:
    su.log_warning("map_3d module not available — skipping 3D demos.")

## Summary

This notebook demonstrated the `siege_utilities.geo.choropleth` and `siege_utilities.reporting.map_3d` modules:

| Function | Use Case |
|----------|----------|
| `create_choropleth()` | Single-variable map from GeoDataFrame |
| `create_choropleth_comparison()` | Multiple variables side by side |
| `create_classified_comparison()` | Compare classification schemes |
| `create_bivariate_choropleth()` | Two variables with 2D color legend (now with magnitude ticks) |
| `create_bivariate_crosstab()` | NxN count table for bivariate bins |
| `create_bivariate_companion_maps()` | Side-by-side monovariate maps with matching bins |
| `verify_bivariate_classification()` | Data-level verification of classification integrity |
| `create_bivariate_analysis()` | One-call orchestrator producing all artifacts |
| `save_map()` | Export to PNG/PDF/SVG |
| `ThreeDMapRenderer` | Interactive 3D maps via pydeck (deck.gl) |
| `create_3d_hexbin()` | Hexagonal binning layer |
| `create_3d_columns()` | Column layer with height proportional to value |

**Bivariate Color Schemes:** `purple_blue`, `teuling`, `blue_red`, `green_orange`

**Classification Schemes** (via mapclassify): `quantiles`, `equal_interval`, `fisher_jenks`, `percentiles`

**3D Map Styles:** `dark`, `light`, `satellite`, `outdoors` (via pydeck/deck.gl)

**Key Color Schemes (cmaps):**
- Sequential: `'Blues'`, `'Greens'`, `'YlOrRd'`, `'Purples'`
- Diverging: `'RdBu'`, `'RdYlGn'`, `'BrBG'`
- See: https://matplotlib.org/stable/gallery/color/colormap_reference.html